# Wine Value-for-Money

*Tasting note → VFM score (0–99)*

**Dataset:** `spawn99/wine-reviews` (~281K reviews, CC BY-NC-SA 4.0)
**Target:** `vfm` = log(points/price) min-max scaled on fixed analytic bounds → integer 0–99
(`utils/vfm.py` — stateless, reproducible from curation constants alone)
**Curation:** points 80–100, price $4–250 (≈91.6% of the market), description 100–2,000 chars

| Part | Content |
|---|---|
| 1 | Curation, VFM derivation, EDA suite, splits, optional push to Hub |
| 2 | Pre-processing: deterministic input assembly (+ optional LLM rewrite ablation) |
| 3 | Evaluation harness, baselines, traditional ML |
| 4 | Deep neural network + frontier zero-shot (composite path) |
| 5 | Fine-tuning a frontier model |
| 6 | Deterministic verdict layer demo (bargain / fair / overpriced) |

- **Target redesigned**
    Rejected raw `points/price` ratio (unstable, non-standard scale); adopted **VFM** = `log(points/price)` min-max scaled on fixed analytic bounds → integer **0–99**, deterministic, stateless, reproducible from curation constants alone (`utils/vfm.py`)

- **Curation**
    `spawn99/wine-reviews`, filtered to points 80–100, **price $4–250** (~91.6% of market), description 100–2,000 chars, deduped on (title, description) — **no subsampling/exclusion of any country** (full dataset used as-is)

- **Preprocessing**
    `TextAssembler` builds `wine.summary` = tasting note + Variety/Country/Province/Region/Winery, all flattened into one text block — **no separate categorical/dummy columns in any text-based model**. Title excluded by default (memorization-leakage risk). Optional Groq Batch API rewrite path built (`WineBatch`) for full-scale LLM rewrite of tasting notes.

- **EDA**
    VFM distribution, points/price/log-price, VFM by country/variety/province/region, and the price–points plane colored by VFM

- **Evaluation harness**
    `Tester`/`evaluate` — MAE ± 95% CI, R², color-coded scatter, **cumulative-error trend chart with shaded 95% CI band**

- **Model ladder built**
    Random/constant baselines → feature LR (top-5 country/variety dummies + note length) → bag-of-words LR → **combined BoW+dummies variants of LR/RF** → residual DNN (`VfmNet`, HashingVectorizer, plain standardization) → frontier LLM zero-shot, **two paths**: `frontier_composite` (model estimates points+price → frozen `compute_vfm`, fair comparison) and `frontier_direct` (cold 0–99 guess, expected weaker — VFM is our construct)

- **Fine-tuning rung — BLOCKED**
    OpenAI is winding down self-serve fine-tuning (May 2026 announcement).

## Part 0 — Setup

In [ ]:
from __future__ import annotations

import json
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

from utils.items import Wine
from utils.loaders import WineLoader
from utils.preprocessor import TextAssembler
from utils.evaluator import evaluate
from utils.deep_neural_network import VfmTrainer
from utils.vfm import compute_vfm, apply_vfm, VFM_LOG_MIN, VFM_LOG_MAX

load_dotenv(override=True)

In [ ]:
# ── Global constants ──────────────────────────────────────────────────────────
SEED: int = 42
TRAIN_FRACTION: float = 0.8
VAL_FRACTION: float = 0.1          # test gets the remainder

FRONTIER_MODEL: str = "gpt-4.1"
FINE_TUNE_BASE_MODEL: str = "gpt-4.1-nano-2025-04-14"  # verify current name in OpenAI docs

client = OpenAI()
print(f"VFM analytic bounds: log_min={VFM_LOG_MIN:.3f}, log_max={VFM_LOG_MAX:.3f}")

## Part 1 — Data Curation

Curation rules live in `utils/parser.py` (points 80–100, **price $4–250**, description 100–2,000 chars, non-empty context fields). Dedup on (title, description) in `utils/loaders.py`. Then `apply_vfm()` derives the target — pure function of (points, price), so it can run before splits with zero leakage.

In [ ]:
# Inspect the raw schema
from datasets import load_dataset
raw = load_dataset("spawn99/wine-reviews")
print(raw)
next(iter(raw[list(raw.keys())[0]]))

In [ ]:
loader = WineLoader()
wines = loader.load()
apply_vfm(wines)
wines[0]

In [ ]:
print(wines[0].full)
print("---")
print(f"points={wines[0].points}, price=${wines[0].price}, VFM={wines[0].vfm}")

### EDA suite

Target first (VFM distribution), then the source quantities (points, price),
then VFM across country / variety / province / region, and the price–points
plane colored by VFM to see exactly what the transform rewards.


In [ ]:
# ── EDA 1: the target — VFM distribution ──────────────────────────────────────
vfm_values = [w.vfm for w in wines]
plt.figure(figsize=(15, 4))
plt.title(f"VFM: avg {np.mean(vfm_values):.1f}, std {np.std(vfm_values):.1f}, "
          f"median {np.median(vfm_values):.0f}")
plt.xlabel("VFM (0-99)"); plt.ylabel("Count")
plt.hist(vfm_values, rwidth=0.8, color="#4A6FA5", bins=range(0, 101, 2))
plt.show()

In [ ]:
# ── EDA 2: source quantities — points and price ───────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

points = [w.points for w in wines]
axes[0].hist(points, rwidth=0.8, color="#5BA4A4", bins=range(80, 102))
axes[0].set_title(f"Points (avg {np.mean(points):.1f})"); axes[0].set_xlabel("Points")

prices = [w.price for w in wines]
axes[1].hist(prices, rwidth=0.8, color="#C4A35B", bins=range(0, 255, 5))
axes[1].set_title(f"Price (median ${np.median(prices):.0f})"); axes[1].set_xlabel("USD")

axes[2].hist(np.log10(prices), rwidth=0.8, color="#C4A35B", bins=40)
axes[2].set_title("log10(Price)")
axes[2].set_xlabel("log10 USD")
plt.show()

In [ ]:
# ── EDA 3: VFM by country (top 12 by volume) ──────────────────────────────────
frame = pd.DataFrame(
    {
        "country": [w.country for w in wines],
        "province": [w.province for w in wines],
        "region": [w.region for w in wines],
        "variety": [w.variety for w in wines],
        "points": [w.points for w in wines],
        "price": [w.price for w in wines],
        "vfm": [w.vfm for w in wines],
    }
)

top_countries = frame["country"].value_counts().head(12).index
by_country = (
    frame[frame["country"].isin(top_countries)]
    .groupby("country")["vfm"]
    .agg(["median", "mean", "count"])
    .sort_values("median")
)
print(by_country)

by_country["median"].plot(kind="barh", figsize=(12, 5), color="#4A6FA5",
                          title="Median VFM by country (top 12 by volume)")
plt.xlabel("Median VFM")
plt.tight_layout()
plt.show()

In [ ]:
# ── EDA 4: VFM by grape variety (top 15 by volume) ────────────────────────────
top_varieties = frame["variety"].value_counts().head(15).index
by_variety = (
    frame[frame["variety"].isin(top_varieties)]
    .groupby("variety")["vfm"]
    .median()
    .sort_values()
)
by_variety.plot(kind="barh", figsize=(12, 6), color="#5BA4A4",
                title="Median VFM by variety (top 15 by volume)")
plt.xlabel("Median VFM")
plt.tight_layout()
plt.show()

In [ ]:
# ── EDA 5: best/worst VFM provinces and regions (min 300 wines) ───────────────
MIN_GROUP: int = 300

by_province = frame.groupby("province")["vfm"].agg(["median", "count"])
by_province = by_province[by_province["count"] >= MIN_GROUP].sort_values("median")
print("Bottom 5 provinces by VFM:\n", by_province.head(5), "\n")
print("Top 5 provinces by VFM:\n", by_province.tail(5), "\n")

by_region = frame[frame["region"] != "Unknown"].groupby("region")["vfm"].agg(["median", "count"])
by_region = by_region[by_region["count"] >= MIN_GROUP].sort_values("median")
combined = pd.concat([by_region.head(7), by_region.tail(7)])
combined["median"].plot(kind="barh", figsize=(12, 6), color="#C47E7E",
                        title=f"Median VFM — 7 worst and 7 best regions (n ≥ {MIN_GROUP})")
plt.xlabel("Median VFM")
plt.tight_layout()
plt.show()

In [ ]:
# ── EDA 6: the price-points plane, colored by VFM ─────────────────────────────
sample_frame = frame.sample(min(20_000, len(frame)), random_state=SEED)
plt.figure(figsize=(12, 6))
scatter = plt.scatter(sample_frame["price"], sample_frame["points"],
                      c=sample_frame["vfm"], cmap="RdYlGn", s=6, alpha=0.5)
plt.colorbar(scatter, label="VFM")
plt.xscale("log")
plt.xlabel("Price USD (log)"); plt.ylabel("Points")
plt.title("What the VFM transform rewards: quality per log-dollar")
plt.show()

In [ ]:
sample: list[Wine] = wines

In [ ]:
# ── Shuffle + splits, fixed seed ──────────────────────────────────────────────
random.seed(SEED)
random.shuffle(sample)
for index, wine in enumerate(sample):
    wine.id = index

n_train = int(len(sample) * TRAIN_FRACTION)
n_val = int(len(sample) * VAL_FRACTION)
train = sample[:n_train]
val = sample[n_train : n_train + n_val]
test = sample[n_train + n_val :]
print(f"train={len(train):,}  val={len(val):,}  test={len(test):,}")

# Sanity: VFM distribution should match across splits (pure function of row data)
for name, split in [("train", train), ("val", val), ("test", test)]:
    values = [w.vfm for w in split]
    print(f"{name}: VFM avg {np.mean(values):.1f}, std {np.std(values):.1f}")


## Part 2 — Pre-processing

Primary path is **deterministic assembly** (`TextAssembler`): tasting note + Variety/Country/Province/Region/Winery. Title excluded by default (`INCLUDE_TITLE=False`) — vintage + winery invite memorization leakage; flip for the ablation. The optional `LLMRewriter`/Batch path tests whether standardizing prose helps. Whichever path trains is the path that must serve at inference.

In [ ]:
assembler = TextAssembler()
assembler.run(train)
assembler.run(val)
assembler.run(test)

print(train[0].prompt)
print("---")
print(test[0].test_prompt())

In [ ]:
# ── Push the curated, split dataset to the Hub (do this once) ────────────────

HUB_DATASET_NAME: str = "gtraskas/wine-vfm-curated"  # set your HF namespace

Wine.push_to_hub(HUB_DATASET_NAME, train, val, test)
print(f"Pushed {len(train):,} train / {len(val):,} val / {len(test):,} test wines to {HUB_DATASET_NAME}")

In [ ]:
wines[0].model_dump()

In [ ]:
# ── OPTIONAL ablation: LLM rewrite on a slice, then re-evaluate later ─────────
# rewriter = LLMRewriter()
# rewriter.run(train[:1000])
# For scale, use utils/batch.py (Batch API, ~50% cheaper; match by custom_id).

In [ ]:
# ── OPTIONAL: push the curated dataset to the Hub (license: CC BY-NC-SA 4.0) ──
# Wine.push_to_hub(HUB_DATASET_NAME, train, val, test)

## Part 3 — Evaluation Harness, Baselines, Traditional ML

One harness (`utils/evaluator.py`): **MAE ± 95% CI and R²** on the same held-out slice, bands green < 5 / orange < 15 on the 0–99 scale. Ladder: random → constant → feature LR → bag-of-words LR → Random Forest → XGBoost.

In [ ]:
def random_vfm(wine: Wine) -> float:
    """Uniform random guess across the VFM scale."""
    return random.uniform(0, 99)

random.seed(SEED)
evaluate(random_vfm, test)

In [ ]:
TRAIN_MEAN_VFM: float = float(np.mean([w.vfm for w in train]))

def constant_vfm(wine: Wine) -> float:
    """Always predict the training-set mean — the honest floor."""
    return TRAIN_MEAN_VFM

evaluate(constant_vfm, test)

In [ ]:
# ── Engineered features + linear regression ───────────────────────────────────
from sklearn.linear_model import LinearRegression

TOP_N_CATEGORY: int = 5

TOP_COUNTRIES: list[str] = (
    pd.Series([w.country for w in train]).value_counts().head(TOP_N_CATEGORY).index.tolist()
)
TOP_VARIETIES: list[str] = (
    pd.Series([w.variety for w in train]).value_counts().head(TOP_N_CATEGORY).index.tolist()
)
print(f"Top countries: {TOP_COUNTRIES}")
print(f"Top varieties: {TOP_VARIETIES}")


def get_features(wine: Wine) -> dict[str, float]:
    """Note length + top-5 country one-hot + top-5 variety one-hot.
    Anything outside the top 5 falls to all-zeros (implicit 'other' bucket).
    NOTE: price/points are excluded — they ARE the label."""
    features = {"note_length": float(len(wine.full))}
    for country in TOP_COUNTRIES:
        features[f"country_{country}"] = float(wine.country == country)
    for variety in TOP_VARIETIES:
        features[f"variety_{variety}"] = float(wine.variety == variety)
    return features


np.random.seed(SEED)
feature_columns = list(get_features(train[0]).keys())
train_frame = pd.DataFrame([get_features(w) for w in train])
train_frame["vfm"] = [w.vfm for w in train]
feature_lr = LinearRegression().fit(train_frame[feature_columns], train_frame["vfm"])
for column, coef in zip(feature_columns, feature_lr.coef_):
    print(f"{column}: {coef:.4f}")


def feature_linear_regression(wine: Wine) -> float:
    """Predict VFM from note length + top-5 country/variety one-hot."""
    row = pd.DataFrame([get_features(wine)])
    return float(feature_lr.predict(row[feature_columns])[0])


evaluate(feature_linear_regression, test)

In [ ]:
# ── Bag of words + linear regression (text only) ──────────────────────────────
from sklearn.feature_extraction.text import CountVectorizer

np.random.seed(SEED)
train_targets = np.array([w.vfm for w in train], dtype=float)
vectorizer = CountVectorizer(max_features=2000, stop_words="english")
train_matrix = vectorizer.fit_transform([w.summary for w in train])
bow_lr = LinearRegression().fit(train_matrix, train_targets)

def bow_linear_regression(wine: Wine) -> float:
    """Predict VFM from bag-of-words over the assembled text."""
    return float(bow_lr.predict(vectorizer.transform([wine.summary]))[0])

evaluate(bow_linear_regression, test)

In [ ]:
# ── Combined features: bag-of-words + top-5 country/variety dummies ──────────
from scipy.sparse import csr_matrix, hstack

def get_dummy_features(wine: Wine) -> dict[str, float]:
    """Top-5 country/variety one-hot — frozen categories from TOP_COUNTRIES/TOP_VARIETIES."""
    features = {}
    for country in TOP_COUNTRIES:
        features[f"country_{country}"] = float(wine.country == country)
    for variety in TOP_VARIETIES:
        features[f"variety_{variety}"] = float(wine.variety == variety)
    return features

dummy_columns = list(get_dummy_features(train[0]).keys())

def to_combined(wines: list[Wine]) -> csr_matrix:
    """BoW text matrix + categorical dummies, hstacked into one sparse matrix."""
    text_features = vectorizer.transform([w.summary for w in wines])
    dummy_rows = csr_matrix(
        pd.DataFrame([get_dummy_features(w) for w in wines])[dummy_columns].values
    )
    return hstack([text_features, dummy_rows]).tocsr()

train_combined = to_combined(train)

In [ ]:
# ── Bag-of-words + dummies, linear regression ─────────────────────────────────
bow_lr_combined = LinearRegression().fit(train_combined, train_targets)

def bow_linear_regression_combined(wine: Wine) -> float:
    return float(bow_lr_combined.predict(to_combined([wine]))[0])

evaluate(bow_linear_regression_combined, test)

In [ ]:
# ── Bag-of-words + dummies, Random Forest (RF_SUBSET cap) ────────────────
from sklearn.ensemble import RandomForestRegressor

RF_SUBSET: int = 30_000
rf_combined_model = RandomForestRegressor(n_estimators=100, random_state=SEED, n_jobs=10)
rf_combined_model.fit(train_combined[:RF_SUBSET], train_targets[:RF_SUBSET])

def random_forest_combined(wine: Wine) -> float:
    return float(rf_combined_model.predict(to_combined([wine]))[0])

evaluate(random_forest_combined, test)

## Part 4 — Deep Neural Network + Frontier LLMa

The DNN lives in `utils/deep_neural_network.py` — residual bag-of-words
regressor, plain standardization. For the frontier, the **composite path** is
primary: the model estimates points and price (quantities that exist in the
world), and the SAME frozen `compute_vfm` maps them to 0–99 — a fair ladder
comparison. A direct "guess the 0–99 score" variant is included for contrast;
expect it to be weaker since VFM is our construct, not world knowledge.


In [ ]:
trainer = VfmTrainer()
trainer.train(train)

In [ ]:
def neural_network(wine: Wine) -> float:
    """Predict VFM with the trained DNN."""
    return trainer.predict(wine)

evaluate(neural_network, test)

In [ ]:
def composite_messages_for(wine: Wine) -> list[dict[str, str]]:
    """Frontier prompt: estimate points and price as JSON."""
    instruction = (
        "Estimate this wine's Wine Enthusiast critic score (80-100 points) "
        "and its retail price in USD from the tasting note and details. "
        'Respond ONLY with JSON: {"points": <int>, "price": <float>}'
    )
    return [{"role": "user", "content": f"{instruction}\n\n{wine.summary}"}]


def frontier_composite(wine: Wine) -> float:
    """Frontier estimates (points, price); frozen compute_vfm maps to 0-99."""
    response = client.chat.completions.create(
        model=FRONTIER_MODEL,
        messages=composite_messages_for(wine),
        max_tokens=40,
    )
    reply = response.choices[0].message.content.replace("```json", "").replace("```", "")
    try:
        estimate = json.loads(reply)
        points = float(np.clip(estimate["points"], 80, 100))
        price = float(np.clip(estimate["price"], 4, 250))
        return float(compute_vfm(points, price))
    except (json.JSONDecodeError, KeyError, TypeError, ValueError):
        return TRAIN_MEAN_VFM  # graceful fallback on malformed output

In [ ]:
print(test[0].summary)
print(frontier_composite(test[0]), "vs truth", test[0].vfm)

In [ ]:
evaluate(frontier_composite, test)

## Part 5 — Fine-Tuning a Frontier Model

OpenAI SFT on the VFM target directly — unlike zero-shot, a fine-tune CAN learn our construct from examples, which is exactly the point of this rung. 200 train / 50 validation, 1 epoch. The assistant turn goes in TRAINING JSONL only.

In [ ]:
FINE_TUNE_TRAIN_SIZE: int = 200
FINE_TUNE_VAL_SIZE: int = 50

fine_tune_train = train[:FINE_TUNE_TRAIN_SIZE]
fine_tune_validation = val[:FINE_TUNE_VAL_SIZE]

In [ ]:
# ── Contrast: ask for the 0-99 score directly (expected weaker) ───────────────
def direct_messages_for(wine: Wine) -> list[dict[str, str]]:
    """Frontier prompt: guess the 0-99 VFM score directly."""
    instruction = (
        "Rate this wine's value-for-money from 0 (worst value) to 99 (best "
        "value), where the score reflects quality relative to price. "
        "Respond only with the number."
    )
    return [{"role": "user", "content": f"{instruction}\n\n{wine.summary}"}]


def frontier_direct(wine: Wine) -> str:
    """Direct 0-99 guess — our construct, not world knowledge."""
    response = client.chat.completions.create(
        model=FRONTIER_MODEL, messages=direct_messages_for(wine), max_tokens=10
    )
    return response.choices[0].message.content

evaluate(frontier_direct, test)

In [ ]:
def training_messages_for(wine: Wine) -> list[dict[str, str]]:
    """User prompt + expected assistant answer — TRAINING JSONL ONLY."""
    return direct_messages_for(wine) + [
        {"role": "assistant", "content": f"{wine.vfm}"}
    ]


def write_jsonl(wines: list[Wine], filename: str) -> None:
    """Write fine-tuning JSONL: one messages object per line."""
    with open(filename, "w", encoding="utf-8") as f:
        for wine in wines:
            f.write(json.dumps({"messages": training_messages_for(wine)}) + "\n")


write_jsonl(fine_tune_train, "fine_tune_train.jsonl")
write_jsonl(fine_tune_validation, "fine_tune_validation.jsonl")

In [ ]:
with open("fine_tune_train.jsonl", "rb") as f:
    train_file = client.files.create(file=f, purpose="fine-tune")
with open("fine_tune_validation.jsonl", "rb") as f:
    validation_file = client.files.create(file=f, purpose="fine-tune")
train_file, validation_file

## OpenAI shut down self-serve fine-tuning entirely, so this section is now a placeholder. The code is still here for reference, but it will not run.

In [ ]:
job = client.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=validation_file.id,
    model=FINE_TUNE_BASE_MODEL,
    seed=SEED,
    hyperparameters={"n_epochs": 1, "batch_size": 1},
    suffix="wine-vfm",
)
job.id

In [ ]:
# Poll until status == 'succeeded'
client.fine_tuning.jobs.retrieve(job.id)


In [ ]:
client.fine_tuning.jobs.list_events(fine_tuning_job_id=job.id, limit=10).data

In [ ]:
fine_tuned_model_name = client.fine_tuning.jobs.retrieve(job.id).fine_tuned_model
fine_tuned_model_name


In [ ]:
def fine_tuned_frontier(wine: Wine) -> str:
    """VFM estimate from the fine-tuned model — inference messages only."""
    response = client.chat.completions.create(
        model=fine_tuned_model_name,
        messages=direct_messages_for(wine),
        max_tokens=7,
    )
    return response.choices[0].message.content

In [ ]:
print(test[0].vfm)
print(fine_tuned_frontier(test[0]))

In [ ]:
evaluate(fine_tuned_frontier, test)